# Chapter 21 — Atomic and Synchronization Operations

Part 8: Advanced Triton operations and production.

**Ladder:** non-atomic read-modify-write race (Method 01) → atomic update with `tl.atomic_add`, `tl.atomic_max`, `tl.atomic_min`, and `tl.atomic_cas` (Method 02) → deterministic segmented reduction (Method 03).

**Thread.**

- **Recurring idea.** An atomic makes one read-modify-write indivisible **per address**; it does not make a floating-point reduction deterministic.
- **Chain.** Non-atomic RMW → lost update → `atomic_add`/`atomic_max`/`atomic_min`/CAS → contention → races → memory ordering → synchronization limits → deterministic segmented reduction.
- **Payoff.** You will choose between an atomic, a partitioned write, and a fixed reduction tree, and say which one is correct and which one is reproducible.

**Study tip.**

- Two programs both execute `y = y + 1` on the same cell, initially `0`.
- Write the six-step interleaving in which both load `0`, both add one, and both store `1`: the final value is `1`, not `2`, so one update was lost.
- Write the interleaving in which program 0 finishes its read-modify-write before program 1 begins: the final value is `2`.
- Every atomic operation forces the second schedule; every deterministic reduction avoids the question altogether.

**Learning objectives**

- Read every atomic operation as one indivisible read-modify-write and state the value it returns.
- State the **atomicity/order rule**: atomicity is per address; ordering of other accesses requires `sem="release"`, `sem="acquire"`, or `sem="acq_rel"`.
- Compute the lost-update interleaving, the contention cost, and the duplicate-id scatter-add by hand.
- Choose the correct identity (`0`, `-inf`, `+inf`, `0` flag) before launching an atomic kernel.
- Quantify contention and reduce it with per-block local accumulation.
- Replace a floating-point atomic reduction with a deterministic two-pass or segmented reduction.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../../part7_construction/ch20_fusion/20_complete.ipynb) · [Next →](../ch22_dot_gemm/22_complete.ipynb)

**Estimated time:** 70–85 min · **Difficulty:** Advanced

**Prerequisites.**

- Chapter 18: grids and boundary masks.
- Chapter 20: fused kernels.
- Chapter 5: pointers, loads, stores, and the read-modify-write pattern.
- No prior lock-free programming is assumed.

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Concept diagrams

- **Thread.** The four diagrams trace one idea at four scales: one cell, two programs, many programs, and a sorted reduction.

### 1. Read-modify-write

```text
non-atomic increment of one cell

  program 0                     memory y
  ---------                     --------
  load  r0 <- y                 0
  add   r0 <- r0 + 1           0
  store y  <- r0                1

  the three steps are separate memory operations;
  another program can run between any two of them.

atomic increment of one cell

  program 0
  ---------
  [ load y, add 1, store y ]    <-- one indivisible transaction
  returns the old value        0
  memory y                     1
```

- **Read-modify-write.** The bracket is the promise: no other atomic on **that address** can be placed between the load and the store. An atomic is not a faster load or store.

### 2. Lost update

```text
time   program 0                  program 1                  y
----   ---------                  ---------                  -
 t0    load  r0 <- y                                        0
 t1                              load  r1 <- y               0
 t2    add   r0 <- 1                                        0
 t3                              add   r1 <- 1               0
 t4    store y  <- r0                                       1
 t5                              store y  <- r1              1   <-- r0 lost

Both programs read the same old value. One store overwrites the other.
The final value is 1 even though two increments were requested.
```

- **Lost update.** Two programs read the same old value; one store overwrites the other, so two increments leave `1`.

### 3. Atomic add contention

```text
P programs                 K distinct addresses
    |   |   |                  |    |         |
    +---+---+------------------+----+---------+
            |                  |              |
            v                  v              v
      one hot address       address 0      address K-1
            |                  |              |
      serialize all P     run in parallel   run in parallel

time ->  [a0][a1][a2] ... [aP-1]        (hot address)
         [a0] [a0] [a0] ...             (distinct addresses, pipelined)

Rule of thumb: serialization is proportional to the number of conflicting
lanes per address, not to the total number of lanes.
```

- **Contention.** One hot address serializes all `P` updates; `K` distinct addresses run in parallel, so serialization tracks conflicts per address.

### 4. Deterministic segmented reduction

```text
ids    = [2, 0, 2, 2, 1]
values = [1, 4, 5, 2, 7]

sort by id (stable)          segment          reduce
-------------------          -------          ------
ids    0  1  2  2  2        id 0: [4]   ->   4
values 4  7  1  5  2        id 1: [7]   ->   7
                            id 2: [1,5,2] -> 8

out = [4, 7, 8]

Every addition happens in a fixed left-to-right order inside a fixed segment.
The result is bitwise reproducible; no atomic is used.
```

- **Segmentation.** A stable sort fixes the segment order, and each segment reduces left to right; the result is bitwise reproducible without an atomic.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, next_pow2, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU, "| triton:", triton.__version__)

# Part I — Mathematics and concepts

- **Connection.** Builds on the Chapter 5 load-modify-store pattern; this part names the indivisible version of that pattern and separates atomicity from ordering.
- **Definition.** An **atomic operation** is a read-modify-write that executes as one indivisible memory transaction.
- **Formula.** For address $p$ and value $v$: $\operatorname{atomicAdd}(p,v): y_{\text{old}}\leftarrow *p,\ *p\leftarrow y_{\text{old}}+v,\ \text{return }y_{\text{old}}$.
- **Atomicity.** The read, the update, and the write cannot be separated by another atomic to the same address; this property is local to one address.
- **Linearizability.** Every atomic operation appears to take effect at a single instant between invocation and return, and all programs agree on that order; GPU atomics are linearizable per address, but the hardware chooses the order.
- **Three questions.** (1) What does one operation do (`atomic_add`, `atomic_max`, `atomic_min`, `atomic_cas`)? (2) What happens when many lanes touch one address (contention, races)? (3) What does the hardware promise about other addresses (ordering, synchronization limits, deterministic alternatives)?

### 21.1 `tl.atomic_add()`

- **Connection.** Builds on the Part I definition of a read-modify-write and on the Chapter 5 load-modify-store skeleton.
- **Rule.** For each active lane, `tl.atomic_add(ptr, val, mask=..., sem=..., scope=...)` executes one indivisible add and returns the pre-update value; a vector call with $L$ lanes performs $L$ independent atomic updates, one per pointer element.
- **Formula.** $y_f=y_0+\sum_{i=0}^{P-1}v_i$ and $\text{return}_i=y_0+\sum_{j\prec i}v_j$, where $j\prec i$ means $j$ was serialized before $i$. The final value is order-independent **only because integer addition is associative and commutative**; the returned value is not.

- **Example.** Let the cell start at $y_0=0$ and let two programs each add $1$; the hardware may choose either atomic order.

| step | atomic order | action | `y` after | value returned |
|---|---|---|---|---|
| 1 | $P_0$ first | `atomic_add(y, 1)` | 1 | 0 |
| 2 | $P_1$ second | `atomic_add(y, 1)` | 2 | 1 |

If the order is reversed, the returned values are $0$ and $1$ in the other order, but the final value is still $0+1+1=2$.

- **Where.** Scatter-add (embedding backward), histograms, bias-gradient accumulation with repeated ids, split-K accumulation, counters, and any “many writers, one output” pattern.
- **Triton.** Initialize the output to the identity `0`, use `int32` for counts, pass the same mask that guards the load, and call `tl.atomic_add(out_ptr + ids, values, mask=mask)`; the output must already exist and be zeroed, and a masked-off lane must not update any address.
- **Failure.** Writing `out[ids] += values` as a load, add, and store lets two lanes with the same id read the old value and lose one update; expecting the returned old values in program order is also wrong because they arrive in hardware order.
- **Bridge.** §21.2 keeps the same read-modify-write shape but replaces addition with an idempotent update.

### 21.2 `tl.atomic_max()`

- **Connection.** Builds on §21.1: the read-modify-write shape is unchanged, but the update is a maximum instead of an add.
- **Rule.** `tl.atomic_max(ptr, val)` atomically replaces the word with the larger of the old value and `val`, then returns the old value; it is the identity-initialized running maximum.
- **Formula.** $*p\leftarrow\max(*p,v)$, and because $\max(\max(a,b),c)=\max(a,\max(b,c))$ with $\max(a,a)=a$, the final value is independent of arrival order.

- **Example.** Initialize $y=-\infty$ and feed three values in any order.

| step | value | `y` after | returned |
|---|---|---|---|
| 1 | 3 | 3 | $-\infty$ |
| 2 | 7 | 7 | 3 |
| 3 | 2 | 7 | 7 |

The final value is always $\max(v_0,v_1,v_2)=7$; only the returned values change with order.

- **Where.** Global maximum of a tensor, maximum logit for a stable softmax, running best score, per-bin maximum (segment extrema), and clamping a shared bound.
- **Triton.** Call `tl.atomic_max(out_ptr + bin, x, mask=mask)`; initialize floating outputs to `-float("inf")` and integer outputs to the smallest representable value, and for a block reduction collapse the tile with `tl.max` before **one** atomic per program.
- **Failure.** Initializing to `0` when every real value is negative keeps the answer at `0`; float atomics lowered through compare-and-swap can be slower than expected, so check the dtype; and `NaN` has no total order, so state a NaN policy before using `atomic_max` on unvalidated data.
- **Bridge.** §21.3 is the mirror image: the same idempotent update, but with a minimum.

### 21.3 `tl.atomic_min()`

- **Connection.** Builds on §21.2: same idempotent read-modify-write, reversed comparison.
- **Rule.** `tl.atomic_min(ptr, val)` atomically replaces the word with the smaller of the old value and `val`, then returns the old value; it is the mirror image of `atomic_max`.
- **Formula.** $*p\leftarrow\min(*p,v)$, so the final value is $\min(v_0,\dots,v_{n-1})$ for every arrival order; only the returned values change.

- **Example.** Initialize $y=+\infty$ and feed $1,-3,2$.

| step | value | `y` after | returned |
|---|---|---|---|
| 1 | 1 | 1 | $+\infty$ |
| 2 | -3 | -3 | 1 |
| 3 | 2 | -3 | -3 |

The final value is $\min(1,-3,2)=-3$ for every arrival order.

- **Where.** Minimum pairwise distance, earliest timestamp, lower bound of a clipping interval, per-bin minimum in a bucketed summary, and the first index of a match (combined with `atomic_min` on the index, not the value).
- **Triton.** Call `tl.atomic_min(out_ptr + bin, x, mask=mask)`; initialize to `+float("inf")` for floats and the largest representable value for integers, and combine with the bin index `ids % NBINS` for a per-bin result.
- **Failure.** Initializing to `0` when every value is positive, reusing a buffer left at `+inf` from a previous launch, or forgetting the bin mask so padded lanes write `+inf` into bin `0`.
- **Bridge.** §21.4 generalizes the update itself: CAS can implement any read-modify-write, including max and min when the hardware lacks them.

### 21.4 Compare-and-swap

- **Connection.** Builds on §21.1–§21.3: instead of a fixed arithmetic update, CAS exposes the compare as part of the atomic transaction.
- **Rule.** In Triton, `old = tl.atomic_cas(ptr, cmp, val)` writes `val` only when the current word equals `cmp`, returns the word it observed, and makes the test and the write one atomic step.
- **Formula.** $\operatorname{CAS}(p,c,v)=v$ when $*p=c$, otherwise $\operatorname{CAS}(p,c,v)=*p$; in both cases it returns $*p_{\text{old}}$.

- **Example — a CAS increment loop.** Two programs both want `y <- y + 1`, starting at `0`.

| step | program | `old` read | `CAS(old, old+1)` | result | `y` |
|---|---|---|---|---|---|
| 1 | $P_0$ | 0 | `CAS(0,1)` | succeeds, returns 0 | 1 |
| 2 | $P_1$ | 0 | `CAS(0,1)` | fails, returns 1 | 1 |
| 3 | $P_1$ | 1 | `CAS(1,2)` | succeeds, returns 1 | 2 |

Program 1 must **reload** after the failed CAS. The loop is:

```text
old = load(p)
while True:
    new = f(old)
    seen = CAS(p, old, new)
    if seen == old: break
    old = seen
```

The retry is why CAS is **lock-free** rather than wait-free: some program always makes progress, but an individual program can starve under adversarial scheduling.

- **Where.** Implementing an arbitrary atomic update when the hardware exposes only CAS, a one-shot flag, a lock, a lock-free stack or queue, and a monotonic counter that must not skip values.
- **Triton.** `tl.atomic_cas(ptr, cmp, val, sem=..., scope=...)` supports 16-, 32-, and 64-bit elements; use integers for flags and counters, and compare float bit patterns through `tl.cast`/`tl.bitcast`, never with `==`, because `NaN != NaN` and `-0.0 == 0.0`.
- **Failure.** **ABA** changes the word $0\to1\to0$ between the read and the CAS, so the CAS succeeds on a state the program never observed; a counter or version tag beside the value fixes it. Other failures are forgetting to reload, spinning without a bound, and assuming a successful CAS orders unrelated memory unless `sem` says so.
- **Bridge.** §21.5 asks what happens when many CAS loops, or many atomics, target the same address at once.

### 21.5 Atomic contention

- **Connection.** Builds on §21.1–§21.4: the operations are correct per address, but their cost depends on how many programs choose the same address.
- **Rule.** Atomics to **distinct** addresses proceed in parallel; atomics to the **same** address serialize at the memory unit that owns it.
- **Formula.** With $P$ updates to one address and service time $\tau$: $T_{\text{hot}}\approx P\tau$. Spread uniformly over $K$ addresses: $T_{\text{spread}}\approx(P/K)\tau$ and speedup $\approx K$. Balls-in-bins fluctuations put the maximum slightly above the mean: $M_K\approx P/K+\sqrt{(2P/K)\ln K}$.

- **Example.** Let $N=2^{20}=1{,}048{,}576$ updates, block size $B=256$, and $K=64$ bins.

| quantity | per-lane atomic | local bins + one atomic per bin |
|---|---|---|
| programs | — | $P=\lceil N/B\rceil=4096$ |
| atomics | $N=1{,}048{,}576$ | $PK=262{,}144$ |
| hottest-address updates | $N$ | $P=4096$ |
| time at $\tau=50$ ns | $52.43$ ms | $0.2048$ ms |

With $\tau=50$ ns the hot-address time drops from $52.43$ ms to $0.2048$ ms, a $256\times$ reduction. The total atomic count falls by a factor $N/(PK)=B/K$: $256$ when $K=1$, and $4$ when $K=64$.

- **Where.** Histograms, embedding gradients where the padding token id repeats, reductions to a single scalar, and any shared counter updated once per element.
- **Triton.** Accumulate locally first with `count = tl.sum((x == bin_id).to(tl.int32), axis=0)` and then `tl.atomic_add(hist_ptr + bin_id, count)`; the local `tl.sum` is a tree inside one program, so only one atomic per bin leaves the program.
- **Failure.** One atomic per lane to one address, or a shared counter inside a loop: both are correct and both can be arbitrarily slower than local accumulation.
- **Bridge.** §21.6 names the failure that appears when the atomic is missing entirely.

### 21.6 Race conditions

- **Connection.** Builds on §21.5: contention is the cost of many correct atomics; a race is the wrong answer when the update is not atomic at all.
- **Rule.** A **data race** exists when two programs access the same memory location, at least one access is a write, and no synchronization orders the accesses.
- **Formula.** A non-atomic increment is $\text{load}\to\text{add}\to\text{store}$; these are three accesses, so the race window is the whole sequence, not one instruction.

- **Example.** The canonical lost update, with the cell initially $0$.

| time | $P_0$ | $P_1$ | `y` |
|---|---|---|---|
| $t_0$ | `r0 = load(y)` | — | 0 |
| $t_1$ | — | `r1 = load(y)` | 0 |
| $t_2$ | `r0 = r0 + 1` | — | 0 |
| $t_3$ | — | `r1 = r1 + 1` | 0 |
| $t_4$ | `store(y, r0)` | — | 1 |
| $t_5$ | — | `store(y, r1)` | 1 |

Two increments were requested; the counter advanced by one. The schedule in which $P_0$ runs all three steps before $P_1$ loads would produce $2$; both schedules are legal, so the program has no defined answer.

- **Where.** Duplicate indices in a scatter, two blocks writing the same output row, a flag set by one block and read by another, and a buffer reused across launches without a dependency.
- **Triton.** First ask whether two programs can write the same address; if not, no atomic is needed. If so, give each address one owner (partition the output), use an atomic, or accumulate per-block partials and merge in a second pass; `+=` on a pointer is never a shortcut for any of these.
- **Failure.** “It worked on my run” is the symptom: the lost update is schedule-dependent, so a test that passes once can fail later. A race on a **different** address is not fixed by making one counter atomic, because atomicity is per address.
- **Bridge.** §21.7 explains why even an atomic update can still expose stale data at another address.

### 21.7 Memory ordering

- **Connection.** Builds on §21.6: removing the race on one address says nothing about when **other** addresses become visible.
- **Rule.** Atomicity and ordering are different promises: **atomicity** makes the read-modify-write of **this address** indivisible, while **ordering** says when **other** loads and stores become visible relative to it.
- **Formula.** `relaxed` < `acquire`/`release` < `acq_rel`; Triton's default is `acq_rel`, not sequentially consistent. Visibility is chosen by `scope`: `cta` < `gpu` < `sys`.

- **Example — publish and consume.** Program 0 writes a payload and then sets a flag; program 1 waits for the flag and reads the payload.

| step | $P_0$ | $P_1$ | visible state |
|---|---|---|---|
| 1 | `store(data, 42)` | — | data may still be in flight |
| 2 | `atomic_xchg(flag, 1, sem="release")` | — | data is ordered before flag |
| 3 | — | `while atomic_add(flag, 0, sem="acquire") == 0` | waits |
| 4 | — | `load(data)` | must see 42 |

If the flag is `"relaxed"` on both sides, step 4 may see the old payload even though it saw the new flag: the flag was atomic, but the data was not ordered.

| `sem` | Promise |
|---|---|
| `"relaxed"` | atomicity only; no ordering of other accesses |
| `"release"` | all prior writes by this program are visible before this atomic |
| `"acquire"` | all later reads by this program see writes released before the matching atomic |
| `"acq_rel"` | both; Triton's default when `sem=None` |

| `scope` | Visibility set |
|---|---|
| `"cta"` | one thread block |
| `"gpu"` | one GPU (the default) |
| `"sys"` | system-wide (GPU plus peer devices / host) |

- **Where.** Producer/consumer flags, lock acquisition, one-shot initialization, and any algorithm where one address announces that another address is ready.
- **Triton.** Use `sem="release"` on the publish and `sem="acquire"` on the consume, `sem="acq_rel"` for a lock or CAS loop, and the narrowest correct scope: `"cta"` intra-block, `"gpu"` (default) cross-block, `"sys"` only for peer or host communication; do not add `sem` by habit, because it can cost more than the atomic itself.
- **Failure.** Assuming the default is sequentially consistent, mixing a `release` publish with a `relaxed` consume, or assuming an atomic on `flag` orders an unrelated `data` address without a matching `acquire`.
- **Bridge.** §21.8 shows the other coordination limit: ordering primitives cannot create a barrier that the programming model does not have.

### 21.8 Synchronization limitations

- **Connection.** Builds on §21.7: memory ordering controls visibility, but it cannot make one program wait for every program in the grid.
- **Rule.** A normal Triton kernel has **no grid-wide barrier**; there is no portable operation inside one kernel that says “wait until every program in the grid has arrived.” `tl.debug_barrier()` is a block-level (CTA) barrier: it synchronizes the threads of one program, not the programs of a grid.
- **Formula.** A grid barrier would require `arrived == num_programs` to hold for all programs simultaneously; Triton exposes no primitive with that guarantee.

- **Example — why a spin barrier deadlocks.** Suppose the grid has more programs than can be resident at once and program 0 does:

```text
store(partial[0], value)
atomic_add(arrived, 1)
while atomic_add(arrived, 0) < num_programs:
    pass
read(partial[1])          # needs program 1 to have run
```

If program 1 is not resident because program 0 occupies its slot, program 1 can never increment `arrived`; program 0 spins forever. This is a **deadlock**, not a slow path.

- **Where.** Split-K reductions, global scans, multi-phase histogram merges, and algorithms copied from a CPU thread-pool mental model.
- **Triton.** Split the algorithm at a kernel boundary: kernel 1 writes each program's partial result, and kernel 2 reads all partials and finishes the reduction. Inside one program use `tl.sum`/`tl.max` over a tile; inside one CTA `tl.debug_barrier()` is appropriate; across programs use atomics only for one-way communication or accumulation, never for waiting.
- **Failure.** A “global barrier” built from an atomic counter and a spin loop, assuming program 0 runs before program 1, or treating `tl.debug_barrier()` as `__syncthreads()` for the whole grid; a second kernel launch is not a workaround here, it is the correct semantics.
- **Bridge.** §21.9 uses the launch boundary to build the deterministic alternative.

### 21.9 Safe reduction alternatives

- **Connection.** Builds on §21.8: if the launch boundary is the only grid-wide barrier, the deterministic reduction must be expressed as fixed per-block work plus a merge.
- **Rule.** A deterministic reduction fixes the **shape of the addition tree** with three portable patterns: in-program reduction (`tl.sum` over the tile), two-pass partials (kernel 1 writes one partial per block, kernel 2 merges in a fixed order), and sort-and-segment (sort by id, find boundaries, reduce each contiguous segment).
- **Formula.** For $N=8$ and `BLOCK=4`: $p_0=x_0+x_1+x_2+x_3$, $p_1=x_4+x_5+x_6+x_7$, and $S=p_0+p_1$. The atomic version computes $S=((x_{\pi(0)}+x_{\pi(1)})+\cdots)+x_{\pi(7)}$ for an arrival permutation $\pi$ chosen by the hardware; both equal $\sum_i x_i$ mathematically, but need not agree bit for bit.

- **Example.** The tree for the two-pass sum is fixed by the block map; the atomic tree changes with arrival order. That is the whole difference between a correct approximate sum and a reproducible result.

- **Where.** Reproducible tests, numerical regression suites, debugging a mismatch that only appears under load, and any reduction whose output feeds a deterministic checksum.
- **Triton.** Write partials to a buffer of length `ceil(N / BLOCK)` and reduce that buffer; for a scatter-add, sort ids on the host, compute segment offsets, and launch one program per segment with `tl.sum` over the contiguous slice. The sort is $O(N\log N)$; the atomics it replaces are $O(N)$ but non-deterministic.
- **Failure.** Calling `atomic_add` in a loop and expecting the same bits every run, using an unstable sort, ignoring empty segments, or reducing the partials with another non-deterministic atomic sum.
- **Bridge.** The next section isolates the floating-point reason this distinction exists.

**Where this leads.** Part II turns these rules into a CPU-checkable schedule model, Part III into atomic kernels, and Part IV into the deterministic reduction described here.

### Why atomics do not fix floating-point order

- **Connection.** Builds on §21.9: the two-pass and segmented trees are deterministic because they fix the parenthesization, while an atomic accumulator does not.
- **Rule.** Atomicity makes each update indivisible; it does **not** make floating-point addition associative.
- **Formula.** In `float32` with $a=10^{20}$, $b=1$, $c=-10^{20}$: $\operatorname{fl}(\operatorname{fl}(a+b)+c)=\operatorname{fl}(10^{20}-10^{20})=0$, but $\operatorname{fl}(\operatorname{fl}(a+c)+b)=\operatorname{fl}(0+1)=1$.
- **Example.** Both results are legal atomic arrival orders: the final cell is $0$ or $1$ depending on which program the hardware serialized first. A large sum behaves the same way: the accumulator commits each addend in arrival order, so the result is one of many parenthesizations.
- **Where.** Use atomics for a correct approximate sum when run-to-run variation is acceptable; use a fixed tree, two-pass partials, or sorted segments when a reproducible bit pattern is required.
- **Failure.** Treating “atomic” as a synonym for “deterministic”; the two properties are orthogonal.
- **Bridge.** The chapter map and worked example below make the atomic and deterministic paths concrete.

**Where this leads.** Part II builds the CPU oracle for the races and CAS steps above; Part III implements the atomic versions; Part IV implements the deterministic alternative.

## Chapter map

| § | Operation | Rule | Failure mode | Kernel/demo |
|---|---|---|---|---|
| **21.1** | `tl.atomic_add()` | $y\mathrel{+}=v$, returns old $y$ | lost update without the atomic | duplicate-id scatter-add |
| **21.2** | `tl.atomic_max()` | $y\leftarrow\max(y,v)$ | initialized to `0` for negative data | global and per-bin extrema |
| **21.3** | `tl.atomic_min()` | $y\leftarrow\min(y,v)$ | initialized to `0` for positive data | global and per-bin extrema |
| **21.4** | Compare-and-swap | write only if $*p=c$ | ABA, no reload after failure | CAS increment counter |
| **21.5** | Atomic contention | $T\approx(\text{conflicts per address})\tau$ | one atomic per lane on a hot address | local-bin histogram |
| **21.6** | Race conditions | load-add-store is not indivisible | schedule-dependent lost update | Method 01 schedule model |
| **21.7** | Memory ordering | `relaxed` < `acquire`/`release` < `acq_rel` | relaxed flag orders nothing else | release/acquire publish |
| **21.8** | Synchronization limitations | no grid-wide barrier in one kernel | spin barrier deadlock | two-pass reduction |
| **21.9** | Safe reduction alternatives | fixed tree or sorted segments | atomic sum is not reproducible | segmented scatter-add |
| **★** | Capstone | local bins + one atomic per bin | contention on the merge | 64-bin histogram |

## Worked numerical example

- **Thread.** Every part below is checked by the next code cell, which only asserts.

### Part A — two programs, one cell

Let $y_0=0$ and let both programs request $+1$.

**Schedule 1 (lost update).** Both load before either stores.

| step | $P_0$ | $P_1$ | `y` |
|---|---|---|---|
| 1 | `r0 = load(y) = 0` | — | 0 |
| 2 | — | `r1 = load(y) = 0` | 0 |
| 3 | `r0 = r0 + 1 = 1` | — | 0 |
| 4 | — | `r1 = r1 + 1 = 1` | 0 |
| 5 | `store(y, r0)` | — | 1 |
| 6 | — | `store(y, r1)` | 1 |

- **Result.** $1$: one increment is lost.

**Schedule 2 (serialized).** $P_0$ finishes before $P_1$ loads.

| step | $P_0$ | $P_1$ | `y` |
|---|---|---|---|
| 1 | `r0 = load(y) = 0` | — | 0 |
| 2 | `r0 = 1` | — | 0 |
| 3 | `store(y, r0)` | — | 1 |
| 4 | — | `r1 = load(y) = 1` | 1 |
| 5 | — | `r1 = 2` | 1 |
| 6 | — | `store(y, r1)` | 2 |

- **Result.** $2$.

**Atomic guarantee.** `atomic_add(y, 1)` removes the interleavings inside the bracket. The only legal orders are $P_0\prec P_1: y:0\to1\to2$ and $P_1\prec P_0: y:0\to1\to2$, so the final value is $2$ and the returned values are $\{0,1\}$ in the chosen order.

### Part B — duplicate-id scatter-add

Let `ids = [2, 0, 2]` and `values = [1, 4, 5]`, with an output of length 3 initialized to zero.

- **Formula.** $\text{out}_0=4$, $\text{out}_1=0$, $\text{out}_2=1+5=6$.
- **Plain store.** It is a race on index 2: the last writer leaves either $1$ or $5$, never $6$.
- **Atomic add.** It leaves $6$ for both arrival orders because $1+5=5+1=6$.

### Part C — CAS loop

Start $y=0$ and let two programs increment through CAS.

| step | program | observed | CAS | outcome | `y` |
|---|---|---|---|---|---|
| 1 | $P_0$ | 0 | `CAS(0,1)` | success | 1 |
| 2 | $P_1$ | 0 | `CAS(0,1)` | failure (returns 1) | 1 |
| 3 | $P_1$ | 1 | `CAS(1,2)` | success | 2 |

- **Result.** $y=2$. The failed CAS is not an error; it is the signal to retry.

### Part D — order-sensitive floating point

With $a=10^{20}$, $b=1$, $c=-10^{20}$ in `float32`: $(a+b)+c=0$ and $(a+c)+b=1$.

- **Result.** Both orders are valid atomic arrival orders; the next cell verifies every number in Parts A–D without printing a table.

In [ ]:
# Part A: simulate the two non-atomic schedules and the atomic guarantee.
def _sim_rmw(y0, schedule):
    """schedule: (program, 'load'|'add'|'store') in time order."""
    regs, y = {}, y0
    for who, op in schedule:
        if op == "load":
            regs[who] = y
        elif op == "add":
            regs[who] = regs[who] + 1
        else:
            y = regs[who]
    return y


lost = _sim_rmw(0, [("p0", "load"), ("p1", "load"),
                    ("p0", "add"), ("p1", "add"),
                    ("p0", "store"), ("p1", "store")])
serial = _sim_rmw(0, [("p0", "load"), ("p0", "add"), ("p0", "store"),
                      ("p1", "load"), ("p1", "add"), ("p1", "store")])
assert lost == 1 and serial == 2
assert 0 + 1 + 1 == 2                     # atomic order does not matter for ints

# Part B: duplicate-id scatter-add.
ids = torch.tensor([2, 0, 2])
values = torch.tensor([1.0, 4.0, 5.0])
atomic_out = torch.zeros(3)
atomic_out.index_add_(0, ids, values)
assert atomic_out.tolist() == [4.0, 0.0, 6.0]
assert values[0] + values[2] == values[2] + values[0] == 6.0

# A plain store keeps only the last writer.
last_writer = torch.zeros(3)
for i, v in zip(ids.tolist(), values.tolist()):
    last_writer[i] = v
assert last_writer[2].item() in (1.0, 5.0) and last_writer[2].item() != 6.0

# Part C: CAS succeeds only against the expected value.
def _cas_once(cell, expected):
    if cell == expected:
        return cell + 1, cell, True
    return cell, cell, False


cell, old, ok = _cas_once(0, 0)
assert (cell, old, ok) == (1, 0, True)
cell, old, ok = _cas_once(cell, 0)
assert (cell, old, ok) == (1, 1, False)
cell, old, ok = _cas_once(cell, 1)
assert (cell, old, ok) == (2, 1, True)

# Part D: float32 addition is not associative.
a = torch.tensor([1e20], dtype=torch.float32)
b = torch.tensor([1.0], dtype=torch.float32)
c = torch.tensor([-1e20], dtype=torch.float32)
assert ((a + b) + c).item() == 0.0
assert ((a + c) + b).item() == 1.0

# Part II — Method 01: host-side race simulation

- **Connection.** Turns the Part I timelines into a CPU oracle before any kernel runs.
- **Models.**
  - `non_atomic_rmw(cell0, schedule)` replays a chosen interleaving of `load`, `add`, and `store`.
  - `scatter_add_nonatomic(ids, values, n)` performs a plain store per contribution, so duplicate ids exhibit last-writer-wins.
  - `scatter_add_serial(ids, values, n)` performs the additions in input order, which is the reference the atomic kernel approximates.
  - `cas_increment_once(cell, expected)` is one CAS attempt; the test uses it to show success, failure, and retry.
- **Why.** Every later claim has a CPU-checkable model; if the model and the kernel disagree, the kernel is wrong or the model is missing a case, and either way the disagreement is a bug rather than a tolerance.

In [ ]:
def non_atomic_rmw(cell0: int, schedule) -> int:
    """Replay (program, op) steps; op is load, add, or store."""
    regs, cell = {}, int(cell0)
    for who, op in schedule:
        if op == "load":
            regs[who] = cell
        elif op == "add":
            regs[who] += 1
        elif op == "store":
            cell = regs[who]
        else:
            raise ValueError(f"unknown op: {op}")
    return cell


def scatter_add_nonatomic(ids: torch.Tensor, values: torch.Tensor, n_out: int) -> torch.Tensor:
    """A plain store per contribution: duplicate ids lose all but the last write."""
    out = torch.zeros(n_out, dtype=values.dtype)
    for i, v in zip(ids.tolist(), values.tolist()):
        out[i] = v
    return out


def scatter_add_serial(ids: torch.Tensor, values: torch.Tensor, n_out: int) -> torch.Tensor:
    """The mathematical scatter-add, executed in the given input order."""
    out = torch.zeros(n_out, dtype=values.dtype)
    for i, v in zip(ids.tolist(), values.tolist()):
        out[i] = out[i] + v
    return out


def cas_increment_once(cell: int, expected: int):
    """One CAS attempt: (new_cell, observed_old, succeeded)."""
    if cell == expected:
        return cell + 1, cell, True
    return cell, cell, False

In [ ]:
def test_method01() -> None:
    lost = non_atomic_rmw(0, [("p0", "load"), ("p1", "load"),
                              ("p0", "add"), ("p1", "add"),
                              ("p0", "store"), ("p1", "store")])
    serial = non_atomic_rmw(0, [("p0", "load"), ("p0", "add"), ("p0", "store"),
                                ("p1", "load"), ("p1", "add"), ("p1", "store")])
    assert lost == 1
    assert serial == 2

    ids = torch.tensor([2, 0, 2, 2])
    vals = torch.tensor([1.0, 4.0, 5.0, 2.0])
    assert scatter_add_nonatomic(ids, vals, 4).tolist() == [4.0, 0.0, 2.0, 0.0]
    assert scatter_add_serial(ids, vals, 4).tolist() == [4.0, 0.0, 8.0, 0.0]

    cell, old, ok = cas_increment_once(0, 0)
    assert (cell, old, ok) == (1, 0, True)
    cell, old, ok = cas_increment_once(cell, 0)
    assert (cell, old, ok) == (1, 1, False)
    cell, old, ok = cas_increment_once(cell, 1)
    assert (cell, old, ok) == (2, 1, True)

    print("method 01: lost update, serial scatter, and CAS retry match the model")


test_method01()

# Part III — Method 02: atomic Triton kernels

- **Connection.** Direct translation of Part I into kernels; each keeps the same seven-line skeleton as every earlier chapter — program id, offsets, mask, load, compute, store — and replaces the final store with one atomic.
- **Scatter-add.** Duplicate ids use `tl.atomic_add(out_ptr + ids, values)`.
- **Global extrema.** One `tl.atomic_max` and one `tl.atomic_min` per program after reducing the tile locally.
- **Per-bin max/min histogram.** `tl.atomic_max`/`tl.atomic_min` indexed by a bin id, one cell per bin.
- **Histogram.** One `tl.atomic_add` per lane to `x % NBINS`.
- **CAS counter.** A `while` loop that retries until the CAS succeeds.
- **Tests.** They run on a GPU, or under the Triton interpreter with `HAS_GPU` forced for the notebook smoke test; on a plain CPU they skip with a clear `SKIP` line. Triton 3.7's interpreter implements all four operations, and if an older interpreter does not, Method 03 is the CPU-safe reference. The interpreter runs programs one at a time, so it validates values and API use, not hardware interleavings: contention and visibility remain hardware properties.

In [ ]:
@triton.jit
def scatter_add_kernel(values_ptr, ids_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    values = tl.load(values_ptr + offs, mask=mask, other=0.0)
    ids = tl.load(ids_ptr + offs, mask=mask, other=0)
    tl.atomic_add(out_ptr + ids, values, mask=mask)


def atomic_scatter_add(values: torch.Tensor, ids: torch.Tensor, n_out: int,
                       BLOCK: int = 256) -> torch.Tensor:
    """out[ids[i]] += values[i], safe for duplicate ids."""
    values = values.contiguous()
    ids = ids.to(torch.int32).contiguous()
    out = torch.zeros(n_out, dtype=values.dtype, device=values.device)
    n = values.numel()
    if n == 0:
        return out
    scatter_add_kernel[(triton.cdiv(n, BLOCK),)](values, ids, out, n, BLOCK=BLOCK)
    return out

In [ ]:
def test_atomic_scatter_add() -> None:
    ids = torch.tensor([2, 0, 2, 5, 0, 2], dtype=torch.int32)
    values = torch.tensor([1.0, 4.0, 5.0, 7.0, -4.0, 2.0])
    out = atomic_scatter_add(values, ids, 8, BLOCK=4)
    ref = torch.zeros(8)
    ref.index_add_(0, ids.long(), values)
    torch.testing.assert_close(out, ref, atol=0.0, rtol=0.0)
    assert out[2].item() == 8.0 and out[0].item() == 0.0
    assert out[5].item() == 7.0 and out[1].item() == 0.0
    print("method 02: duplicate-id scatter-add matches the reference")


run_if_gpu(test_atomic_scatter_add)

In [ ]:
@triton.jit
def atomic_extrema_kernel(x_ptr, max_ptr, min_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    safe = tl.where(mask, x, -float("inf"))
    tl.atomic_max(max_ptr, tl.max(safe, axis=0))
    safe = tl.where(mask, x, float("inf"))
    tl.atomic_min(min_ptr, tl.min(safe, axis=0))


@triton.jit
def atomic_bin_extrema_kernel(x_ptr, bins_ptr, max_ptr, min_ptr, n_elements,
                              BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    bins = tl.load(bins_ptr + offs, mask=mask, other=0)
    tl.atomic_max(max_ptr + bins, x, mask=mask)
    tl.atomic_min(min_ptr + bins, x, mask=mask)


def atomic_extrema(x: torch.Tensor, BLOCK: int = 256):
    mx = torch.full((1,), -float("inf"), dtype=x.dtype, device=x.device)
    mn = torch.full((1,), float("inf"), dtype=x.dtype, device=x.device)
    n = x.numel()
    atomic_extrema_kernel[(triton.cdiv(n, BLOCK),)](x, mx, mn, n, BLOCK=BLOCK)
    return mx.item(), mn.item()


def atomic_bin_extrema(x: torch.Tensor, bins: torch.Tensor, nbins: int, BLOCK: int = 256):
    mx = torch.full((nbins,), -float("inf"), dtype=x.dtype, device=x.device)
    mn = torch.full((nbins,), float("inf"), dtype=x.dtype, device=x.device)
    n = x.numel()
    atomic_bin_extrema_kernel[(triton.cdiv(n, BLOCK),)](
        x, bins.to(torch.int32), mx, mn, n, BLOCK=BLOCK
    )
    return mx, mn

In [ ]:
def test_atomic_extrema() -> None:
    x = torch.tensor([-7.0, 3.0, 2.0, -1.0, 9.0, 0.0, 9.0])
    mx, mn = atomic_extrema(x, BLOCK=4)
    assert mx == 9.0 and mn == -7.0

    bins = torch.tensor([0, 0, 0, 1, 1, 2, 2], dtype=torch.int32)
    bmx, bmn = atomic_bin_extrema(x, bins, 3, BLOCK=8)
    assert bmx.tolist() == [3.0, 9.0, 9.0]
    assert bmn.tolist() == [-7.0, -1.0, 0.0]
    print("method 02: global and per-bin atomic extrema match the data")


run_if_gpu(test_atomic_extrema)

In [ ]:
@triton.jit
def atomic_histogram_kernel(x_ptr, hist_ptr, n_elements, NBINS: tl.constexpr,
                            BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0)
    tl.atomic_add(hist_ptr + (x % NBINS), 1, mask=mask)


def atomic_histogram(x: torch.Tensor, nbins: int = 8, BLOCK: int = 256) -> torch.Tensor:
    hist = torch.zeros(nbins, dtype=torch.int32, device=x.device)
    n = x.numel()
    atomic_histogram_kernel[(triton.cdiv(n, BLOCK),)](
        x, hist, n, NBINS=nbins, BLOCK=BLOCK
    )
    return hist

In [ ]:
def test_atomic_histogram() -> None:
    torch.manual_seed(0)
    x = torch.randint(0, 1000, (1000,), dtype=torch.int32)
    got = atomic_histogram(x, nbins=8, BLOCK=128)
    ref = torch.bincount((x.long() % 8), minlength=8).to(torch.int32)
    assert torch.equal(got, ref)
    assert int(got.sum()) == x.numel()
    print("method 02: atomic histogram matches bincount")


run_if_gpu(test_atomic_histogram)

In [ ]:
@triton.jit
def cas_increment_kernel(cell_ptr, old_ptr):
    old = tl.load(cell_ptr)
    new = old + 1
    seen = tl.atomic_cas(cell_ptr, old, new)
    while seen != old:
        old = seen
        new = old + 1
        seen = tl.atomic_cas(cell_ptr, old, new)
    tl.store(old_ptr + tl.program_id(0), seen)


def cas_increment_counter(n_programs: int):
    """n_programs programs each add 1 through a CAS retry loop."""
    cell = torch.zeros(1, dtype=torch.int32)
    old = torch.empty(n_programs, dtype=torch.int32)
    cas_increment_kernel[(n_programs,)](cell, old)
    return int(cell.item()), old

In [ ]:
def test_cas_counter() -> None:
    n = 64
    final, old = cas_increment_counter(n)
    assert final == n
    assert sorted(old.tolist()) == list(range(n))
    print("method 02: CAS retry loop reaches exactly n increments")


run_if_gpu(test_cas_counter)

# Part IV — Method 03: deterministic segmented reduction

- **Connection.** Answers the question Method 02 leaves open: how do we get the same bits every run?
- **Two-pass full reduction.** Kernel 1 reduces each block and writes one partial; kernel 2 (or a host `sum`) reduces the partials in a fixed order: $p_b=\sum_{i=bB}^{bB+B-1}x_i$, $S=\sum_{b=0}^{P-1}p_b$, $P=\lceil N/B\rceil$.
- **Sorted segmented scatter-add.** Sort the ids, find the unique ids and segment lengths, and reduce each contiguous segment with a fixed tree: $\text{order}=\operatorname{argsort}(\text{ids})$ and $\text{out}_j=\sum_{m:\,\text{ids}[m]=j}v_m$.
- **Payoff.** Both remove the atomic from the accumulation path entirely.

### Contention and traffic model

- **Formula.** With $N$ contributions, block size $B$, $P=\lceil N/B\rceil$ programs, $K$ bins, element size $s$, and atomic service time $\tau$: $Q_{\text{atomic}}=2s(\#\text{atomics})$ and $\#\text{naive}/\#\text{local}=N/(PK)\approx B/K$.
- **Serialization.** The hottest address sees $N$ updates in the naive version and at most $P$ in the local-bin version, so $T_{\text{naive}}/T_{\text{local}}\approx N/P\approx B$.
- **Example.** For $N=2^{20}$, $B=256$, $K=64$, $s=4$: $P=4096$, naive atomics $=1{,}048{,}576$, local atomics $=262{,}144$ (a $4\times$ traffic reduction), hottest-address serialization drops from $1{,}048{,}576$ to $4096$ (a $256\times$ reduction), and atomic traffic drops from $8.39$ MB to $2.10$ MB.
- **Sort cost.** The sort-based variant pays $O(N\log N)$ comparisons and $O(N)$ extra traffic for the permutation, but its additions are fixed.

| Variant | Atomics | Hot-address updates | Deterministic |
|---|---|---|---|
| per-lane atomic | $N$ | $N$ | no |
| local bins + one atomic per bin | $PK$ | $P$ | no |
| two-pass partials | 0 | 0 | yes |
| sort + segmented reduction | 0 | 0 | yes |

In [ ]:
def segmented_scatter_add_reference(values: torch.Tensor, ids: torch.Tensor,
                                    n_out: int) -> torch.Tensor:
    """Deterministic duplicate-id scatter: stable sort + contiguous segments."""
    values = torch.as_tensor(values)
    ids = torch.as_tensor(ids)
    out = torch.zeros(n_out, dtype=values.dtype)
    if values.numel() == 0:
        return out

    order = torch.argsort(ids, stable=True)
    sorted_ids = ids[order]
    sorted_values = values[order]
    unique, counts = torch.unique_consecutive(sorted_ids, return_counts=True)

    start = 0
    for uid, count in zip(unique.tolist(), counts.tolist()):
        segment = sorted_values[start:start + count]
        out[uid] = segment.sum()          # fixed left-to-right segment tree
        start += count
    return out

In [ ]:
@triton.jit
def block_sum_kernel(x_ptr, partial_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(partial_ptr + pid, tl.sum(x, axis=0))


@triton.jit
def atomic_sum_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.atomic_add(out_ptr, tl.sum(x, axis=0))


@triton.jit
def segment_sum_kernel(values_ptr, offsets_ptr, out_ptr, BLOCK: tl.constexpr):
    seg = tl.program_id(0)
    start = tl.load(offsets_ptr + seg)
    end = tl.load(offsets_ptr + seg + 1)
    offs = start + tl.arange(0, BLOCK)
    mask = offs < end
    values = tl.load(values_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + seg, tl.sum(values, axis=0))


def atomic_sum(x: torch.Tensor, BLOCK: int = 256):
    out = torch.zeros(1, dtype=x.dtype, device=x.device)
    atomic_sum_kernel[(triton.cdiv(x.numel(), BLOCK),)](x, out, x.numel(), BLOCK=BLOCK)
    return out[0]


def two_pass_sum(x: torch.Tensor, BLOCK: int = 256):
    n = x.numel()
    parts = torch.empty(triton.cdiv(n, BLOCK), dtype=x.dtype, device=x.device)
    block_sum_kernel[(parts.numel(),)](x, parts, n, BLOCK=BLOCK)
    return parts.sum()


def segmented_scatter_add_triton(values: torch.Tensor, ids: torch.Tensor,
                                 n_out: int) -> torch.Tensor:
    values = torch.as_tensor(values)
    ids = torch.as_tensor(ids)
    out = torch.zeros(n_out, dtype=values.dtype, device=values.device)
    if values.numel() == 0:
        return out

    order = torch.argsort(ids, stable=True)
    sorted_values = values[order].contiguous()
    sorted_ids = ids[order]
    unique, counts = torch.unique_consecutive(sorted_ids, return_counts=True)
    offsets = torch.cat([
        torch.zeros(1, dtype=torch.int64),
        torch.cumsum(counts, dim=0),
    ]).to(torch.int32).contiguous()

    segment_out = torch.empty(unique.numel(), dtype=values.dtype, device=values.device)
    block = next_pow2(int(counts.max().item()))
    segment_sum_kernel[(unique.numel(),)](sorted_values, offsets, segment_out, BLOCK=block)
    out[unique.long()] = segment_out
    return out

In [ ]:
def test_deterministic_reduction() -> None:
    torch.manual_seed(1)
    ids = torch.randint(0, 7, (11,), dtype=torch.int32)
    values = torch.randn(11)

    ref = segmented_scatter_add_reference(values, ids, 7)
    assert torch.equal(ref, segmented_scatter_add_reference(values, ids, 7))
    serial = scatter_add_serial(ids, values, 7)
    torch.testing.assert_close(ref, serial, atol=1e-6, rtol=1e-6)

    x = torch.linspace(-1.0, 1.0, 10003)
    a = atomic_sum(x, BLOCK=256)
    b = two_pass_sum(x, BLOCK=256)
    exact = x.sum()
    torch.testing.assert_close(a, exact, atol=1e-3, rtol=1e-4)
    torch.testing.assert_close(b, exact, atol=1e-3, rtol=1e-4)
    assert torch.equal(b, two_pass_sum(x, BLOCK=256))

    seg = segmented_scatter_add_triton(values, ids, 7)
    torch.testing.assert_close(seg, ref, atol=1e-6, rtol=1e-6)
    print("method 03: two-pass and segmented reductions are reproducible")


run_if_gpu(test_deterministic_reduction)

In [ ]:
N, BLOCK, NBINS, ITEMSIZE = 1 << 20, 256, 64, 4
programs = triton.cdiv(N, BLOCK)
naive_atomics = N
local_atomics = programs * NBINS

assert programs == 4096
assert naive_atomics == 1_048_576
assert local_atomics == 262_144
assert naive_atomics // local_atomics == 4
assert naive_atomics // programs == 256

tau_ns = 50.0
t_naive_ms = naive_atomics * tau_ns / 1e6
t_local_ms = programs * tau_ns / 1e6
assert abs(t_naive_ms - 52.4288) < 1e-3
assert abs(t_local_ms - 0.2048) < 1e-9
assert abs(2 * ITEMSIZE * naive_atomics / 1e6 - 8.388608) < 1e-6
assert abs(2 * ITEMSIZE * local_atomics / 1e6 - 2.097152) < 1e-6
print("contention model: 256x less serialization at the hot address")

## Capstone — a 64-bin histogram with local bins

- **Connection.** Combines the whole chapter: one tile per program, local counting, and one atomic per bin.
- **Rule.** Each program reads one tile, counts how many lanes fall into each bin, and performs **one** atomic add per bin; the number of atomics is $PK$, not $N$, and each bin receives $P$ atomics instead of $N/K$ on average or $N$ on a hot bin.
- **Why integer.** Integer addition is associative and the count fits comfortably in `int32`, so the result is bitwise reproducible even though the merge uses atomics; floating-point bin weights would be approximate in the last bits, exactly the distinction Part I draws.
- **Baseline.** A per-lane atomic histogram is the baseline; the local-bin version is `O(NBINS)` work per tile and `NBINS` atomics per program, so for `NBINS=64` and `BLOCK=256` it trades $256$ per-lane atomics for $64$ per-program atomics.

In [ ]:
@triton.jit
def local_hist_kernel(x_ptr, hist_ptr, n_elements, NBINS: tl.constexpr,
                      BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=-1)
    for bin_id in tl.static_range(0, NBINS):
        count = tl.sum((x == bin_id).to(tl.int32), axis=0)
        tl.atomic_add(hist_ptr + bin_id, count)


def local_histogram(x: torch.Tensor, nbins: int = 64, BLOCK: int = 256) -> torch.Tensor:
    hist = torch.zeros(nbins, dtype=torch.int32, device=x.device)
    n = x.numel()
    local_hist_kernel[(triton.cdiv(n, BLOCK),)](
        x, hist, n, NBINS=nbins, BLOCK=BLOCK
    )
    return hist

In [ ]:
def test_capstone_local_histogram() -> None:
    torch.manual_seed(2)
    x = torch.randint(0, 64, (10_000,), dtype=torch.int32)
    got = local_histogram(x, nbins=64, BLOCK=128)
    ref = torch.bincount(x.long(), minlength=64).to(torch.int32)
    assert torch.equal(got, ref)
    assert int(got.sum()) == x.numel()

    programs = triton.cdiv(x.numel(), 128)
    assert programs * 64 < x.numel()
    print("capstone: local-bin histogram matches bincount")


run_if_gpu(test_capstone_local_histogram)

## Memory ordering and synchronization limitations

- **Connection.** Collects the three assumptions behind the rules in §21.7–§21.8 that cause the most production bugs.

### 1. There is no global barrier in a normal kernel

- **Rule.** Triton exposes no grid-wide synchronization primitive; `tl.debug_barrier()` is block-level.
- **Example.** A spin loop that waits for a counter to reach the grid size can deadlock because the producer program may not be resident while the consumer spins.
- **Fix.** The portable barrier is a **kernel launch**: write partials in kernel 1 and consume them in kernel 2.

### 2. An atomic does not order unrelated accesses

- **Rule.** An atomic on `flag` orders only `flag` unless `sem` says otherwise.
- **Default.** `acq_rel` with `scope="gpu"`, which is safe but not free; a relaxed atomic is appropriate only when no other address depends on it.
- **Example.** A release publish pairs with an acquire consume; a release with a relaxed consume is a bug even though both operations are atomic.

### 3. Block-level synchronization is not grid-level synchronization

- **Rule.** Within one program all lanes execute the same instruction stream, so a tile reduction is already synchronized; across programs there is no such guarantee.
- **Failure.** Do not use `tl.debug_barrier()` to coordinate two programs, and do not assume that program 0 runs first, last, or at the same time as program 1.

| Assumption | Reality | Portable fix |
|---|---|---|
| “The atomic waits for everyone.” | It serializes updates to one address. | second kernel launch |
| “Atomic on flag publishes data.” | Only with `release`/`acquire`. | matching `sem` pair |
| “Program order is grid order.” | Hardware order is unspecified. | partition outputs or reduce in fixed tree |
| “`tl.debug_barrier()` is grid-wide.” | It is CTA-wide. | launch boundary |
| “Relaxed is enough because it is atomic.” | Atomicity ≠ visibility. | `release`/`acquire` or `acq_rel` |

- **Check.** The next cell verifies the sem strings and a release/acquire pair; the kernel runs only where Triton can execute, and the CPU assertions always run.

In [ ]:
_SEMANTICS = ("relaxed", "acquire", "release", "acq_rel")
_SCOPES = ("cta", "gpu", "sys")
assert _SEMANTICS.index("relaxed") < _SEMANTICS.index("acquire") < _SEMANTICS.index("acq_rel")
assert "seq_cst" not in _SEMANTICS          # Triton 3.x does not expose seq_cst
assert set(_SCOPES) == {"cta", "gpu", "sys"}


@triton.jit
def release_publish_kernel(flag_ptr):
    tl.atomic_add(flag_ptr, 1, sem="release", scope="gpu")


@triton.jit
def acquire_observe_kernel(flag_ptr, out_ptr):
    value = tl.atomic_add(flag_ptr, 0, sem="acquire", scope="gpu")
    tl.store(out_ptr, value)


def test_memory_ordering() -> None:
    flag = torch.zeros(1, dtype=torch.int32)
    out = torch.empty(1, dtype=torch.int32)
    release_publish_kernel[(1,)](flag)
    acquire_observe_kernel[(1,)](flag, out)
    assert int(flag.item()) == 1
    assert int(out.item()) == 1
    print("memory ordering: release/acquire pair observes the published flag")


run_if_gpu(test_memory_ordering)

## Pitfalls

| Pitfall | What happens | Fix |
|---|---|
| Non-atomic read-modify-write | duplicate ids lose all but one update | `tl.atomic_add`, partition, or two-pass |
| Using `+=` on a pointer | it is a load, add, and store, not one transaction | spell out the atomic |
| Atomic sum for reproducibility | arrival order changes the floating-point tree | two-pass or segmented reduction |
| Duplicate ids in a plain store | last writer wins, silently | detect collisions or use an atomic |
| One atomic per lane on a hot address | serialization proportional to $N$ | local bins, one atomic per bin per block |
| `atomic_max` initialized to `0` | all-negative data returns `0` | initialize to `-inf` or the type minimum |
| `atomic_min` initialized to `0` | all-positive data returns `0` | initialize to `+inf` or the type maximum |
| CAS without reload | the retry reuses a stale expected value | reload `seen` after failure |
| CAS ABA | a value returns to its old state and the CAS succeeds | version tag or monotonic counter |
| Relaxed flag | data published before the flag may be invisible | `release` publish + `acquire` consume |
| Spin barrier across programs | deadlock when the producer is not resident | second kernel launch |
| Atomics for a full reduction | correct but slow and non-deterministic | `tl.sum` + partials |
| Masked atomic without mask | padded lanes write the identity into bin 0 | pass the same mask |
| Floating `atomic_max`/`atomic_min` on NaN | undefined ordering; NaN may be ignored | define an explicit NaN policy |

## Self-check

1. Write the six-step interleaving that turns two increments of `0` into `1`.
2. Why is `y = y + 1` three memory operations, and what exactly does `atomic_add` make indivisible?
3. Two programs call `atomic_add(y, 1)` on `y=0`. What are the two possible return pairs, and what is the final value?
4. What identity do you use to initialize `atomic_max`, `atomic_min`, and an integer histogram? Why is `0` wrong for the first two in general?
5. State the contention formula for $P$ updates to one address and for $P$ updates spread over $K$ addresses. Which one does a local-bin histogram improve?
6. A CAS loop observes `0`, computes `1`, and the CAS returns `1`. What should the loop do next, and why is not reloading a bug?
7. Give an ABA schedule and one fix. Why does a plain counter not fix it?
8. What is the difference between `sem="relaxed"` and `sem="acquire"`? Which one pairs with `sem="release"`?
9. Why is there no safe grid-wide spin barrier in a normal Triton kernel?
10. Your atomic sum changes in the last bit between runs. Is the kernel wrong? What do you change if the test requires identical bits?

## Summary

- `tl.atomic_add`, `atomic_max`, `atomic_min`, and `atomic_cas` make one read-modify-write indivisible **per address** and return the old value.
- Atomicity removes lost updates; it does not make floating-point addition associative, and it does not order unrelated accesses.
- The atomicity/order rule: atomicity is local; ordering requires an explicit `sem` pair (`release`/`acquire`) and a suitable `scope`.
- Contention is proportional to the number of conflicting lanes per address; local accumulation reduces both the atomic count and the hot-address serialization.
- There is no grid-wide barrier inside a kernel; the launch boundary is the barrier, and `tl.debug_barrier()` is CTA-wide only.
- Determinism comes from a fixed reduction tree: two-pass partials or a sorted segmented reduction. Atomics are for correctness under concurrency, not for reproducibility.
- Initialize every atomic output to the identity, mask every access, and ask first whether two programs can write the same address at all.

**Where this leads.** The exercises below test the same chain end to end: write the lost update, choose an atomic or an owner, quantify contention, and replace the atomic with a fixed tree when the result must be reproducible.

## Exercises

1. **Lost update by hand.** Enumerate all six interleavings of two non-atomic increments. Which ones produce `1`, and which produce `2`? Verify with `non_atomic_rmw`.
2. **Return values.** Extend the scatter-add kernel to store the old value of every atomic. Show that the returned values are a permutation of the prefix sums, not necessarily in program order.
3. **Extrema initialization.** Run `atomic_extrema` on all-negative and all-positive tensors with the wrong identity (`0`) and explain both failures.
4. **CAS lock.** Implement a one-bit spin lock with `atomic_cas`. Explain why a lock held by a non-resident program deadlocks, and why Triton discourages it.
5. **ABA.** Construct a two-program schedule where `0 -> 1 -> 0` lets a stale CAS succeed. Add a version tag and show that the stale CAS now fails.
6. **Contention.** Measure or model the naive and local-bin histograms for `N=2^22`, `BLOCK=256`, and `NBINS` in `{8, 64, 256}`. Which configuration makes `PK > N`?
7. **Deterministic scatter.** Replace `segmented_scatter_add_triton` with a two-level version that sorts ids once and launches one program per segment. Compare its output bit pattern with the atomic version on `float32`.
8. **Two-pass traffic.** Derive the extra bytes read and written by the two-pass sum for `N=2^20`, `BLOCK=256`, `float32`, and compare with the atomic sum.
9. **Scope.** Change the release/acquire pair to `scope="cta"`. Does the test still pass? Under what deployment would it be wrong?
10. **Ownership.** Take a scatter-add whose ids are known to be unique and delete every atomic. Prove that no two programs write the same address, then measure the difference against the atomic version.

In [ ]:
print("Running CPU tests ...")
test_method01()

print("Running GPU/interpreter tests (skipped without a device) ...")
for fn in (
    test_atomic_scatter_add,
    test_atomic_extrema,
    test_atomic_histogram,
    test_cas_counter,
    test_deterministic_reduction,
    test_capstone_local_histogram,
    test_memory_ordering,
):
    run_if_gpu(fn)